# Maia Corpus v9 — Gutenberg Finalization Engine v0.12
Processamento offline integral. Originais somente leitura. A política brasileira conservadora não equipara domínio público nos EUA a domínio público no Brasil. Evidência bibliográfica de autoria não comprova a edição ou tradução. Sem evidência positiva da versão exata, o texto permanece em quarentena. Não há revisão manual obrigatória nem downloads. A meta de 200M tokens é aproximada até existir o tokenizer definitivo.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv,json,re,hashlib,os,collections,unicodedata,time,inspect
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
SRC=ROOT/'gutenberg_v11_offline_rights/rights_inventory_latest.csv'
OUT=ROOT/'gutenberg_v12_finalization'
OUT.mkdir(exist_ok=True)
csv.field_size_limit(100_000_000)
LANGS=('pt','en','es')
TARGET={'pt':60_000_000,'en':80_000_000,'es':60_000_000}
BATCH=100
POLICY='BR_conservative_exact_edition_v1'
# Opcional: evidências positivas já existentes, vinculadas ao SHA do texto exato.
# JSONL: sha256, jurisdiction=BR, basis, evidence_url, evidence_text,
# exact_edition_verified=true, training_permitted=true, all_contributors_verified=true.
# Não gerar tais registros a partir de datas de autores ou licença do dataset.
EVIDENCE=ROOT/'verified_edition_rights.jsonl'
with SRC.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.DictReader(f))
assert len(rows)==39340 and all(k in rows[0] for k in ('sha256','language','source','v11_status','book_id'))
print('Inventário:',len(rows),'candidatos:',sum(r['v11_status']=='CANDIDATE_PUBLIC_DOMAIN' for r in rows))
print('Tamanhos anteriores disponíveis:',sum(bool(r.get('original_characters')) for r in rows))


Mounted at /content/drive
Inventário: 39340 candidatos: 17737
Tamanhos anteriores disponíveis: 39339


## 1. Reconciliação com manifestos e política de direitos
Os caminhos são reconstruídos a partir dos manifestos v0.6/v0.8, não inferidos de títulos. Evidências externas opcionais são declarações previamente verificadas pelo responsável pelo corpus, não provas verificadas por este notebook. Sem elas não haverá aprovação automática definitiva.

In [2]:
def sha(data):return hashlib.sha256(data).hexdigest()
def atomic(path,data):
    tmp=path.with_name(path.name+'.part');tmp.write_bytes(data);os.replace(tmp,path)
def dump(path,obj):atomic(path,json.dumps(obj,ensure_ascii=False,indent=2).encode())
manifest={}; issues=[]; fingerprint=hashlib.sha256(SRC.read_bytes())
for version in ('v06','v08'):
    base=ROOT/('gutenberg_'+version)
    for lang in LANGS:
        p=base/'manifests'/f'{lang}.jsonl'
        raw=p.read_bytes();fingerprint.update(raw)
        for i,line in enumerate(raw.splitlines(),1):
            try:r=json.loads(line)
            except Exception:issues.append({'manifest':str(p),'line':i,'reason':'invalid_json'});continue
            if r.get('status')!='saved':continue
            h=r.get('sha256','')
            if not re.fullmatch('[0-9a-f]{64}',h):issues.append({'manifest':str(p),'line':i,'reason':'invalid_sha'});continue
            key=(version,lang,h)
            if key in manifest and manifest[key][0].get('book_id')!=r.get('book_id'):issues.append({'key':key,'reason':'conflicting_id'})
            manifest[key]=(r,base/'quarantine/books'/lang/(h+'.txt'))
rights=collections.defaultdict(list)
if EVIDENCE.exists():
    raw=EVIDENCE.read_bytes();fingerprint.update(raw)
    for line in raw.splitlines():
        e=json.loads(line);rights[e['sha256']].append(e)
print('Manifestos:',len(manifest),'problemas:',len(issues),'evidências externas:',len(rights))
dump(OUT/'manifest_issues.json',issues)
assert not issues,'Manifestos inconsistentes; corrigir antes de publicar corpus.'
def rights_decision(r):
    ev=rights.get(r['sha256'],[])
    valid=[e for e in ev if e.get('jurisdiction')=='BR' and e.get('basis') in ('public_domain_exact_edition','explicit_training_license') and all(e.get(k) is True for k in ('exact_edition_verified','training_permitted','all_contributors_verified')) and str(e.get('evidence_url','')).startswith('https://') and str(e.get('evidence_text','')).strip()]
    if len(ev)==1 and len(valid)==1:
        if valid[0]['basis']=='explicit_training_license' or r['v11_status']=='CANDIDATE_PUBLIC_DOMAIN':return 'ELIGIBLE_ATTESTED',valid[0]
    return 'QUARANTINE',{'reason':r.get('v11_reason','unknown'),'missing':'verified_exact_edition_and_contributor_rights_BR'}


Manifestos: 39340 problemas: 0 evidências externas: 0


## 2. Auditoria integral e limpeza
Delimitadores ausentes não provam truncamento; a limpeza é bloqueada nesses casos. A identificação de idioma usa heurística de palavras frequentes, não um classificador validado. A deduplicação é exata após normalização, não semântica. Checkpoints por lote; falhas de leitura são reprocessadas na próxima execução.

In [3]:
START=re.compile(r'^\s*\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG[^\n]*?\*\*\*\s*$',re.I|re.M)
END=re.compile(r'^\s*\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG[^\n]*?\*\*\*\s*$',re.I|re.M)
STOP={'pt':set('que de do da não uma para com os as em por se'.split()),'en':set('the and of to in that was with for is he she'.split()),'es':set('que de del la los las una para con por no el'.split())}
def clean(txt):
    a=START.search(txt[:50000]); offset=max(0,len(txt)-100000);z=END.search(txt[offset:])
    if not a or not z or offset+z.start()<=a.end():return None,'missing_or_invalid_boundaries'
    body=unicodedata.normalize('NFC',txt[a.end():offset+z.start()].replace('\r\n','\n').replace('\r','\n')).strip()
    if len(body)<1000 or '\ufffd' in body or '\x00' in body:return None,'short_or_encoding_artifact'
    return body,'ok'
def language_hint(body):
    words=re.findall(r'[^\W\d_]+',body[:100000].lower()); counts={l:sum(w in s for w in words) for l,s in STOP.items()}
    rank=sorted(counts,key=counts.get,reverse=True)
    return rank[0] if counts[rank[0]]>=30 and counts[rank[0]]>1.3*counts[rank[1]] else 'uncertain'
def audit(r):
    h=r['sha256'];lang=r['language'];key=(r['source'],lang,h)
    out={k:r.get(k,'') for k in ('sha256','language','source','book_id','v11_title','v11_authors','v11_status')}
    decision,evidence=rights_decision(r);out.update(rights=decision,evidence=evidence,integrity='failed',quality='not_checked')
    if key not in manifest:return dict(out,error='manifest_missing')
    rec,path=manifest[key];out['path']=str(path)
    try:
        raw=path.read_bytes();txt=raw.decode('utf-8','strict')
        out.update(bytes=len(raw),characters=len(txt),actual_sha256=sha(raw))
        if out['actual_sha256']!=h or (rec.get('bytes') is not None and len(raw)!=int(rec['bytes'])) or (rec.get('characters') is not None and len(txt)!=int(rec['characters'])):return dict(out,error='manifest_integrity_mismatch')
        out['integrity']='ok'
        body,quality=clean(txt);out['quality']=quality
        if body is not None:
            out.update(clean_sha256=sha(body.encode()),clean_chars=len(body),words=len(re.findall(r'\S+',body)),language_hint=language_hint(body),proxy_tokens=(len(body)+3)//4)
            if out['language_hint']!=lang:out['quality']='language_uncertain_or_mismatch'
        return out
    except Exception as e:return dict(out,error=type(e).__name__+': '+str(e))
assert clean('*** START OF THE PROJECT GUTENBERG EBOOK X ***\n'+'abc '*500+'\n*** END OF THE PROJECT GUTENBERG EBOOK X ***')[1]=='ok'
assert clean('abc '*500)[0] is None
fingerprint.update((POLICY+inspect.getsource(clean)+inspect.getsource(audit)+inspect.getsource(rights_decision)+inspect.getsource(language_hint)).encode())
RUN=OUT/fingerprint.hexdigest()[:20];RUN.mkdir(exist_ok=True)
print('Execução:',RUN,flush=True)


Execução: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9/gutenberg_v12_finalization/630dd1fe79b2feaf5f56


In [ ]:
all_results=[];started=time.monotonic()
for start in range(0,len(rows),BATCH):
    p=RUN/f'audit_{start:06d}.json';batch=rows[start:start+BATCH]
    old=[]
    if p.exists():
        try:old=json.loads(p.read_text())
        except Exception:pass
    if len(old)==len(batch) and all(x.get('sha256')==r['sha256'] for x,r in zip(old,batch)):
        result=[x if x.get('integrity')=='ok' else audit(r) for x,r in zip(old,batch)]
    else:result=[audit(r) for r in batch]
    dump(p,result);all_results.extend(result)
    print(f'AUDIT {len(all_results)}/{len(rows)} | íntegros {sum(x["integrity"]=="ok" for x in all_results)} | {time.monotonic()-started:.0f}s',flush=True)
assert len(all_results)==len(rows)


AUDIT 100/39340 | íntegros 100 | 35s
AUDIT 200/39340 | íntegros 200 | 68s
AUDIT 300/39340 | íntegros 300 | 102s
AUDIT 400/39340 | íntegros 400 | 131s
AUDIT 500/39340 | íntegros 500 | 160s
AUDIT 600/39340 | íntegros 599 | 355s
AUDIT 700/39340 | íntegros 699 | 388s
AUDIT 800/39340 | íntegros 799 | 421s
AUDIT 900/39340 | íntegros 899 | 453s
AUDIT 1000/39340 | íntegros 999 | 486s
AUDIT 1100/39340 | íntegros 1099 | 518s
AUDIT 1200/39340 | íntegros 1199 | 550s
AUDIT 1300/39340 | íntegros 1299 | 582s
AUDIT 1400/39340 | íntegros 1399 | 615s
AUDIT 1500/39340 | íntegros 1499 | 647s
AUDIT 1600/39340 | íntegros 1599 | 679s
AUDIT 1700/39340 | íntegros 1699 | 710s
AUDIT 1800/39340 | íntegros 1799 | 746s
AUDIT 1900/39340 | íntegros 1899 | 779s
AUDIT 2000/39340 | íntegros 1999 | 810s
AUDIT 2100/39340 | íntegros 2099 | 842s
AUDIT 2200/39340 | íntegros 2199 | 873s
AUDIT 2300/39340 | íntegros 2299 | 905s
AUDIT 2400/39340 | íntegros 2399 | 937s
AUDIT 2500/39340 | íntegros 2499 | 968s
AUDIT 2600/39340 | ín

## 3. Seleção e exportação
Meta provisória PT 30%, EN 40%, ES 30%, ajustável. Não preencher déficits com obras não elegíveis. Shards JSONL de até 100 livros, com proveniência. A seleção é determinística por hash. Antes de exportar, o texto é relido e os hashes são conferidos novamente. A publicação só ocorre após concluir todos os shards.

In [ ]:
import shutil
export=RUN/'export';export.mkdir(exist_ok=True)
selected=[];seen=set();totals=collections.Counter();stats={}
for lang in LANGS:
    group=[x for x in all_results if x['language']==lang]
    candidates=[x for x in group if x['v11_status']=='CANDIDATE_PUBLIC_DOMAIN' and x['integrity']=='ok']
    pool=[x for x in group if x['rights']=='ELIGIBLE_ATTESTED' and x['integrity']=='ok' and x['quality']=='ok']
    for x in sorted(pool,key=lambda x:x['clean_sha256']):
        if x['clean_sha256'] in seen or totals[lang]>=TARGET[lang]:continue
        seen.add(x['clean_sha256']);selected.append(x);totals[lang]+=x['proxy_tokens']
    stats[lang]={'books':len(group),'integrity_ok':sum(x['integrity']=='ok' for x in group),'candidate_books':len(candidates),'candidate_raw_chars':sum(x['characters'] for x in candidates),'candidate_clean_chars':sum(x.get('clean_chars',0) for x in candidates),'candidate_clean_proxy_tokens':sum(x.get('proxy_tokens',0) for x in candidates),'eligible_quality_books':len(pool),'selected_proxy_tokens':totals[lang],'target_proxy_tokens':TARGET[lang]}
# Separate export identity includes selection configuration.
export=export/sha(json.dumps({'target':TARGET,'selected':[x['sha256'] for x in selected]},sort_keys=True).encode())[:16];export.mkdir(exist_ok=True)
shards=[]
for start in range(0,len(selected),100):
    p=export/f'books_{start//100:05d}.jsonl';tmp=p.with_suffix('.part')
    with tmp.open('w',encoding='utf-8') as f:
        for x in selected[start:start+100]:
            raw=Path(x['path']).read_bytes();assert sha(raw)==x['sha256'],'Original mudou'
            body,q=clean(raw.decode('utf-8'));assert q=='ok' and sha(body.encode())==x['clean_sha256']
            f.write(json.dumps({'text':body,'language':x['language'],'source_dataset':'manu/project_gutenberg','book_id':x['book_id'],'original_sha256':x['sha256'],'clean_sha256':x['clean_sha256'],'rights_evidence':x['evidence']},ensure_ascii=False)+'\n')
    os.replace(tmp,p);shards.append({'file':str(p.relative_to(RUN)),'sha256':sha(p.read_bytes()),'books':min(100,len(selected)-start)})
    print('SHARD',len(shards),flush=True)
summary={'books':len(all_results),'integrity':dict(collections.Counter(x['integrity'] for x in all_results)),'quality':dict(collections.Counter(x['quality'] for x in all_results)),'rights':dict(collections.Counter(x['rights'] for x in all_results)),'selected_books':len(selected),'selected_proxy_tokens':sum(totals.values()),'target_proxy_tokens':sum(TARGET.values()),'by_language':stats,'shards':shards,'policy':POLICY,'status':'EXPORT_READY' if selected else 'BLOCKED_NO_ELIGIBLE_TEXTS'}
for name,group in [('audit_records',all_results),('quarantine',[x for x in all_results if x['rights']!='ELIGIBLE_ATTESTED' or x['integrity']!='ok' or x['quality']!='ok']),('selected',selected)]:
    atomic(RUN/(name+'.jsonl'),''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in group).encode())
dump(RUN/'summary.json',summary);dump(OUT/'latest_run.json',{'run':str(RUN),'summary':str(RUN/'summary.json')})
print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)
if not selected:print('BLOQUEIO: nenhuma edição com evidência positiva suficiente; nenhum shard publicado.')


## 4. Limitações e retomada
Reexecute desde o início para retomar os lotes. Registros íntegros em cache pressupõem originais imutáveis; para nova auditoria integral, remova apenas os checkpoints audit_*.json da execução correspondente. O exportador verifica novamente os originais selecionados. A estimativa de tokens usa quatro caracteres por token e não substitui o MaiaTokenizer. A ausência de marcadores pode refletir pré-processamento do dataset, não corrupção. Não há detector de duplicação aproximada nem comprovação jurídica independente. Se não houver evidência de direitos da edição, o resultado útil será a auditoria e a quantificação dos candidatos, sem corpus liberado.

In [ ]:

# ============================================================
# MAIA CORPUS v9 — MULTILINGUAL TOKEN ESTIMATION
# Complemento do Gutenberg Finalization Engine v0.12
# ============================================================

import json
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_4_0/maia-corpus-v9"
)

OUT = ROOT / "gutenberg_v12_finalization"

# Localizar automaticamente a última execução concluída
latest = json.loads(
    (OUT / "latest_run.json").read_text(encoding="utf-8")
)

SUMMARY_PATH = Path(latest["summary"])

summary = json.loads(
    SUMMARY_PATH.read_text(encoding="utf-8")
)

# Caracteres por token: mínimo, central e máximo
RATIOS = {
    "pt": (3.0, 3.25, 3.5),
    "en": (3.8, 4.00, 4.2),
    "es": (3.5, 3.75, 4.0),
}

TARGETS = {
    "pt": 60_000_000,
    "en": 80_000_000,
    "es": 60_000_000,
}

results = []

for lang, (low, central, high) in RATIOS.items():

    stats = summary["by_language"][lang]

    raw_chars = int(stats["candidate_raw_chars"])
    clean_chars = int(stats["candidate_clean_chars"])

    tokens_min = int(clean_chars / high)
    tokens_est = int(clean_chars / central)
    tokens_max = int(clean_chars / low)

    target = TARGETS[lang]

    results.append({
        "language": lang,
        "candidate_books": stats["candidate_books"],
        "raw_characters": raw_chars,
        "clean_characters": clean_chars,
        "chars_per_token": central,
        "tokens_min": tokens_min,
        "tokens_estimated": tokens_est,
        "tokens_max": tokens_max,
        "target_tokens": target,
        "potential_coverage_pct": round(
            100 * tokens_est / target, 2
        ),
        "potential_surplus": max(0, tokens_est - target),
        "potential_deficit": max(0, target - tokens_est),
    })

df = pd.DataFrame(results)

# ============================================================
# Estatísticas consolidadas
# ============================================================

total_min = int(df["tokens_min"].sum())
total_est = int(df["tokens_estimated"].sum())
total_max = int(df["tokens_max"].sum())
total_target = sum(TARGETS.values())

report = {
    "method": "language_specific_characters_per_token",
    "status": "PROVISIONAL_ESTIMATE",
    "scope": "integrity_verified_public_domain_candidates",
    "source_summary": str(SUMMARY_PATH),
    "ratios": RATIOS,
    "candidate_books": int(df["candidate_books"].sum()),
    "raw_characters": int(df["raw_characters"].sum()),
    "clean_characters": int(df["clean_characters"].sum()),
    "tokens_min": total_min,
    "tokens_estimated": total_est,
    "tokens_max": total_max,
    "target_tokens": total_target,
    "potential_coverage_pct": round(
        100 * total_est / total_target, 2
    ),
    "potential_coverage_with_language_quotas_pct": round(
        100 * sum(
            min(r["tokens_estimated"], r["target_tokens"])
            for r in results
        ) / total_target, 2
    ),
    "by_language": results,
    "requires_final_tokenizer": True,
    "does_not_grant_training_rights": True,
}

# ============================================================
# Persistência dos resultados no Google Drive
# ============================================================

REPORT_DIR = SUMMARY_PATH.parent

JSON_PATH = REPORT_DIR / "token_estimation_multilingual.json"
CSV_PATH = REPORT_DIR / "token_estimation_multilingual.csv"

JSON_PATH.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

df.to_csv(CSV_PATH, index=False)

# ============================================================
# Apresentação dos resultados
# ============================================================

print("=" * 70)
print("MAIA CORPUS v9 — MULTILINGUAL TOKEN ESTIMATION")
print("=" * 70)

display(df.style.format({
    "candidate_books": "{:,.0f}",
    "raw_characters": "{:,.0f}",
    "clean_characters": "{:,.0f}",
    "chars_per_token": "{:.2f}",
    "tokens_min": "{:,.0f}",
    "tokens_estimated": "{:,.0f}",
    "tokens_max": "{:,.0f}",
    "target_tokens": "{:,.0f}",
    "potential_coverage_pct": "{:.2f}%",
    "potential_surplus": "{:,.0f}",
    "potential_deficit": "{:,.0f}",
}))

print(f"\nLivros candidatos: {report['candidate_books']:,}")
print(f"Caracteres limpos: {report['clean_characters']:,}")

print(f"\nEstimativa mínima:  {total_min:,} tokens")
print(f"Estimativa central: {total_est:,} tokens")
print(f"Estimativa máxima:  {total_max:,} tokens")

print(f"\nMeta literária: {total_target:,} tokens")
print(f"Cobertura potencial: {report['potential_coverage_pct']:.2f}%")

print(
    "Cobertura respeitando cotas por idioma: "
    f"{report['potential_coverage_with_language_quotas_pct']:.2f}%"
)

print("\nATENÇÃO:")
print("Estimativa provisória, não contagem real de tokens.")
print("Candidatos não equivalem a textos liberados para treinamento.")
print("Recalcular após o treinamento do MaiaTokenizer-65k.")

print("\nRelatórios salvos:")
print(JSON_PATH)
print(CSV_PATH)

print("=" * 70)
